# Fundamentals of Machine Learning — Exercise 7

## Gradient Boosting: Learning by Correcting Previous Errors

In Exercise 6, a Random Forest trained many trees largely **in parallel** and combined their predictions. Gradient boosting follows a different strategy:

> **Build a sequence of small models, where each new model focuses on what the current ensemble still gets wrong.**

This exercise moves from one transparent residual update to practical classification with validation curves and early stopping.

The classroom workflow remains:

> **question → prediction → computation → visualisation → interpretation → verification**

### Learning outcomes

After completing this exercise, you should be able to:

- explain boosting as an additive sequence of weak learners,
- calculate and interpret residuals in a simple regression example,
- explain the role of `learning_rate` and `n_estimators`,
- distinguish training improvement from validation improvement,
- recognise underfitting and overfitting from staged curves,
- compare a single tree, Random Forest, Gradient Boosting, and Histogram Gradient Boosting,
- use validation data to select the number of boosting stages,
- keep the test set untouched until the final model decision,
- justify a boosting configuration under a simple resource constraint,
- use an AI chatbot to question an already written diagnosis rather than generate it.

**Recommended duration:** 100–120 minutes.  
The final model-selection memo may be completed as homework.

## How to work in this exercise

Most routine code is prepared. Your main tasks are to:

1. predict what a new boosting stage should change,
2. calculate one small example manually,
3. change only clearly marked settings,
4. interpret training and validation curves,
5. justify a model decision using evidence,
6. verify that the test set was not used during model selection.

A high training score is not the goal. The goal is a model that generalises to unseen data.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import (
    load_breast_cancer,
    make_friedman1,
    make_moons,
)
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42

## 1. The central idea: improve the current prediction

For regression with squared error, a convenient first model is a constant:

\[
F_0(x) = \bar{y}.
\]

The residual for observation \(i\) is

\[
r_i = y_i - F_0(x_i).
\]

A small tree is trained to predict these residuals. The ensemble is then updated:

\[
F_1(x) = F_0(x) + \eta h_1(x),
\]

where:

- \(h_1(x)\) is the first small tree,
- \(\eta\) is the learning rate.

### Predict before computing

If the current model underpredicts an observation, is its residual positive or negative?  
Should the next tree increase or decrease the prediction there?

In [ ]:
x_toy = np.array([0, 1, 2, 3, 4, 5], dtype=float)
y_toy = np.array([1.0, 1.4, 1.8, 4.2, 4.6, 5.0])

initial_prediction = np.repeat(y_toy.mean(), len(y_toy))
initial_residual = y_toy - initial_prediction

toy_table = pd.DataFrame({
    'x': x_toy,
    'observed y': y_toy,
    'initial prediction': initial_prediction,
    'residual y - prediction': initial_residual,
})

display(toy_table.round(3))
print('Initial MSE:', round(mean_squared_error(y_toy, initial_prediction), 4))

### Activity 1 — Read the residuals

Without fitting a tree yet:

1. Which observations need their prediction increased?
2. Which observations need their prediction decreased?
3. Around which value of `x` would a one-split tree probably separate the residuals?
4. Why is the residual not the final target variable of the original problem?

### Fit one decision stump to the residuals

A stump is a decision tree with `max_depth=1`. It can create only one split.

In [ ]:
LEARNING_RATE = 0.50

first_stump = DecisionTreeRegressor(
    max_depth=1,
    random_state=RANDOM_STATE,
)
first_stump.fit(x_toy.reshape(-1, 1), initial_residual)

stump_correction = first_stump.predict(x_toy.reshape(-1, 1))
updated_prediction = initial_prediction + LEARNING_RATE * stump_correction
updated_residual = y_toy - updated_prediction

first_stage_table = pd.DataFrame({
    'x': x_toy,
    'observed y': y_toy,
    'before': initial_prediction,
    'residual before': initial_residual,
    'stump correction': stump_correction,
    'after update': updated_prediction,
    'residual after': updated_residual,
})

display(first_stage_table.round(3))
print('MSE before:', round(mean_squared_error(y_toy, initial_prediction), 4))
print('MSE after: ', round(mean_squared_error(y_toy, updated_prediction), 4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(x_toy, y_toy, s=75, label='Observed values')
plt.plot(x_toy, initial_prediction, marker='o', label='Initial constant model')
plt.plot(x_toy, updated_prediction, marker='o', label='After one stump')
plt.xlabel('x')
plt.ylabel('y')
plt.title('One boosting update')
plt.legend()
plt.show()

### Stop and explain

Complete the argument:

> The first stump does not predict `y` directly. It predicts __________.  
> Multiplying by the learning rate causes __________.  
> After the update, the residuals change because __________.

What would happen if `LEARNING_RATE = 0`?  
What risk might arise if the learning rate were extremely large?

## 2. Repeating the process creates an additive model

The next function repeats the same three operations:

1. compute residuals,
2. fit a shallow tree,
3. add a shrunken correction.

Read the loop and identify where each operation occurs.

In [ ]:
def fit_simple_gradient_boosting(x, y, n_stages=10, learning_rate=0.20):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)

    prediction = np.repeat(y.mean(), len(y)).astype(float)
    history = [prediction.copy()]
    trees = []

    for _ in range(n_stages):
        residual = y - prediction

        tree = DecisionTreeRegressor(
            max_depth=1,
            random_state=RANDOM_STATE,
        )
        tree.fit(x, residual)

        correction = tree.predict(x)
        prediction = prediction + learning_rate * correction

        trees.append(tree)
        history.append(prediction.copy())

    return trees, history

_, toy_history = fit_simple_gradient_boosting(
    x_toy,
    y_toy,
    n_stages=20,
    learning_rate=0.20,
)

stage_mse = [
    mean_squared_error(y_toy, prediction)
    for prediction in toy_history
]

pd.DataFrame({
    'stage': range(len(stage_mse)),
    'training MSE': stage_mse,
}).head(10).round(4)

In [ ]:
selected_stages = [0, 1, 3, 10, 20]

plt.figure(figsize=(9, 5))
plt.scatter(x_toy, y_toy, s=80, label='Observed values')

for stage in selected_stages:
    plt.plot(
        x_toy,
        toy_history[stage],
        marker='o',
        label=f'Stage {stage}',
    )

plt.xlabel('x')
plt.ylabel('y')
plt.title('The additive prediction after selected stages')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(range(len(stage_mse)), stage_mse, marker='o')
plt.xlabel('Boosting stage')
plt.ylabel('Training MSE')
plt.title('Training error on the toy data')
plt.show()

### Activity 2 — What becomes more complex?

1. Each individual stump is simple. Why can their sum represent a more complex function?
2. Why does training error usually decrease as stages are added?
3. Why does decreasing training error not guarantee better performance on new data?

## 3. Learning rate and number of trees must be considered together

A smaller learning rate makes each update more conservative, but usually requires more trees. A larger learning rate learns faster, but may reach overfitting earlier.

We now compare several configurations on a nonlinear regression dataset.

In [ ]:
X_reg, y_reg = make_friedman1(
    n_samples=800,
    n_features=6,
    noise=1.2,
    random_state=RANDOM_STATE,
)

X_reg_train, X_reg_valid, y_reg_train, y_reg_valid = train_test_split(
    X_reg,
    y_reg,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

regression_configs = {
    'slow: lr=0.03, 350 trees': {
        'learning_rate': 0.03,
        'n_estimators': 350,
    },
    'balanced: lr=0.10, 180 trees': {
        'learning_rate': 0.10,
        'n_estimators': 180,
    },
    'aggressive: lr=0.50, 80 trees': {
        'learning_rate': 0.50,
        'n_estimators': 80,
    },
}

regression_models = {}
regression_curve_rows = []

plt.figure(figsize=(10, 6))

for label, params in regression_configs.items():
    model = GradientBoostingRegressor(
        max_depth=2,
        random_state=RANDOM_STATE,
        **params,
    )
    model.fit(X_reg_train, y_reg_train)
    regression_models[label] = model

    valid_mse_curve = [
        mean_squared_error(y_reg_valid, prediction)
        for prediction in model.staged_predict(X_reg_valid)
    ]

    best_stage = int(np.argmin(valid_mse_curve) + 1)
    best_valid_mse = float(np.min(valid_mse_curve))

    regression_curve_rows.append({
        'configuration': label,
        'best validation stage': best_stage,
        'best validation MSE': best_valid_mse,
        'final validation MSE': valid_mse_curve[-1],
    })

    plt.plot(
        range(1, len(valid_mse_curve) + 1),
        valid_mse_curve,
        label=label,
    )

plt.xlabel('Number of boosting stages')
plt.ylabel('Validation MSE')
plt.title('Validation error across boosting stages')
plt.legend()
plt.show()

pd.DataFrame(regression_curve_rows).round(3)

### Activity 3 — Diagnose the curves

For each configuration, identify:

- whether learning is slow or fast,
- approximately where the best validation result occurs,
- whether later stages continue to help or begin to hurt.

Then answer:

> Why is it incomplete to state that “a learning rate of 0.03 is better than 0.10” without mentioning the number of trees?

## 4. From simple stumps to a nonlinear classification boundary

Gradient boosting for classification optimises a classification loss rather than directly fitting class labels. The additive model is transformed into probabilities.

We keep each tree shallow and increase only the number of boosting stages.

### Predict before running

How should the decision boundary change from 1 to 100 stages?

In [ ]:
X_moons, y_moons = make_moons(
    n_samples=600,
    noise=0.25,
    random_state=RANDOM_STATE,
)

X_moon_train, X_moon_test, y_moon_train, y_moon_test = train_test_split(
    X_moons,
    y_moons,
    test_size=0.30,
    stratify=y_moons,
    random_state=RANDOM_STATE,
)

moon_stages = [1, 5, 30, 100]
moon_models = {}
moon_rows = []

for n_stages in moon_stages:
    model = GradientBoostingClassifier(
        n_estimators=n_stages,
        learning_rate=0.10,
        max_depth=1,
        random_state=RANDOM_STATE,
    )
    model.fit(X_moon_train, y_moon_train)
    moon_models[n_stages] = model

    probability = model.predict_proba(X_moon_test)[:, 1]
    prediction = (probability >= 0.5).astype(int)

    moon_rows.append({
        'stages': n_stages,
        'test accuracy': accuracy_score(y_moon_test, prediction),
        'test ROC-AUC': roc_auc_score(y_moon_test, probability),
        'test log loss': log_loss(y_moon_test, probability),
    })

pd.DataFrame(moon_rows).round(3)

In [ ]:
def plot_decision_surface(axis, model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    probability = model.predict_proba(grid)[:, 1].reshape(xx.shape)

    axis.contourf(xx, yy, probability, levels=20, alpha=0.35, cmap='coolwarm')
    axis.contour(xx, yy, probability, levels=[0.5], colors='black', linewidths=1.2)
    axis.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=22, alpha=0.75)
    axis.set_title(title)
    axis.set_xticks([])
    axis.set_yticks([])

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

for axis, n_stages in zip(axes, moon_stages):
    plot_decision_surface(
        axis,
        moon_models[n_stages],
        X_moon_test,
        y_moon_test,
        f'{n_stages} stage(s)',
    )

plt.tight_layout()
plt.show()

### Stop and explain

1. How can many axis-aligned stump decisions create a curved-looking boundary?
2. Which change in the table suggests that probability quality improved, even when accuracy changed only slightly?
3. Why should we not automatically select the model with the most stages?

## 5. Real-data comparison: breast cancer diagnosis

We reuse the Breast Cancer dataset so that the new method can be compared directly with models from Exercise 6.

The split has three roles:

- **training set:** fit model parameters,
- **validation set:** choose model and number of stages,
- **test set:** final audit after the choice is fixed.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data.copy()
y = cancer.target.copy()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

print('Training rows:  ', len(X_train))
print('Validation rows:', len(X_valid))
print('Test rows:      ', len(X_test))
print('Positive-class rate:', round(float(y.mean()), 3))

### Predict before fitting

1. Which model should have the largest training–validation gap: a single deep tree, Random Forest, or Gradient Boosting with shallow trees?
2. Does a tree-based model require feature standardisation for its split rules?
3. Why must the test set remain unused in the next comparison?

In [ ]:
comparison_models = {
    'Decision tree': DecisionTreeClassifier(
        max_depth=None,
        random_state=RANDOM_STATE,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=1,
        random_state=RANDOM_STATE,
    ),
    'Histogram Gradient Boosting': HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=15,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=RANDOM_STATE,
    ),
}


def evaluate_classifier(model, X_train, y_train, X_eval, y_eval):
    model.fit(X_train, y_train)

    train_probability = model.predict_proba(X_train)[:, 1]
    eval_probability = model.predict_proba(X_eval)[:, 1]
    train_prediction = (train_probability >= 0.5).astype(int)
    eval_prediction = (eval_probability >= 0.5).astype(int)

    return {
        'train accuracy': accuracy_score(y_train, train_prediction),
        'validation accuracy': accuracy_score(y_eval, eval_prediction),
        'validation F1': f1_score(y_eval, eval_prediction),
        'validation ROC-AUC': roc_auc_score(y_eval, eval_probability),
        'validation log loss': log_loss(y_eval, eval_probability),
    }

validation_rows = []

for name, model in comparison_models.items():
    metrics = evaluate_classifier(
        model,
        X_train,
        y_train,
        X_valid,
        y_valid,
    )
    validation_rows.append({'model': name, **metrics})

validation_results = pd.DataFrame(validation_rows).set_index('model')
validation_results.round(3)

### Activity 4 — Compare by more than accuracy

For each model, comment on:

- training versus validation accuracy,
- validation ROC-AUC,
- validation log loss,
- model complexity or training strategy.

Then write a preliminary recommendation. Do **not** use the test set yet.

**Preliminary recommendation:**

## 6. Select the number of stages using validation loss

`GradientBoostingClassifier` exposes predictions after every stage. This allows us to ask:

> At which stage is validation log loss lowest?

The training curve will usually continue decreasing. The validation curve may eventually flatten or rise.

In [ ]:
gb_full = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=1,
    random_state=RANDOM_STATE,
)
gb_full.fit(X_train, y_train)

train_log_loss_curve = [
    log_loss(y_train, probability)
    for probability in gb_full.staged_predict_proba(X_train)
]
valid_log_loss_curve = [
    log_loss(y_valid, probability)
    for probability in gb_full.staged_predict_proba(X_valid)
]

BEST_STAGE = int(np.argmin(valid_log_loss_curve) + 1)

print('Best validation stage:', BEST_STAGE)
print('Best validation log loss:', round(min(valid_log_loss_curve), 4))

plt.figure(figsize=(10, 5.5))
plt.plot(
    range(1, len(train_log_loss_curve) + 1),
    train_log_loss_curve,
    label='Training log loss',
)
plt.plot(
    range(1, len(valid_log_loss_curve) + 1),
    valid_log_loss_curve,
    label='Validation log loss',
)
plt.axvline(BEST_STAGE, linestyle='--', label=f'Best validation stage: {BEST_STAGE}')
plt.xlabel('Boosting stage')
plt.ylabel('Log loss')
plt.title('Training and validation loss across stages')
plt.legend()
plt.show()

### Activity 5 — Diagnose the stage curve

Complete:

> Training loss continues to __________ because __________.  
> Validation loss reaches its minimum near stage __________.  
> Adding many more stages after this point risks __________.  
> The stage was selected using the __________ set, not the test set.

Why is the minimum validation loss a more defensible stopping point than the minimum training loss?

### Guided AI study activity — challenge your diagnosis

First write your own diagnosis of the curve and your proposed number of stages.

Then use this prompt:

```text
Act as a machine-learning tutor.
Do not choose the model or rewrite my answer.
Ask me exactly three short questions, one at a time:

1. What evidence in the training and validation curves supports my diagnosis?
2. How do learning_rate and n_estimators interact in this result?
3. Have I kept the test set untouched until the final decision?

Wait for my answer after each question.
```

Close the chat and record only:

**My original diagnosis:**  

**One useful AI question:**  

**One sentence I revised:**  

**Why the revision is more accurate:**

## 7. Final retraining and one test-set audit

After choosing the stage count, combine training and validation data, fit the selected model, and evaluate the untouched test set once.

In [ ]:
selected_gb = GradientBoostingClassifier(
    n_estimators=BEST_STAGE,
    learning_rate=0.05,
    max_depth=1,
    random_state=RANDOM_STATE,
)
selected_gb.fit(X_train_full, y_train_full)

test_probability = selected_gb.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

final_test_result = pd.Series({
    'test accuracy': accuracy_score(y_test, test_prediction),
    'test F1': f1_score(y_test, test_prediction),
    'test ROC-AUC': roc_auc_score(y_test, test_probability),
    'test log loss': log_loss(y_test, test_probability),
})

final_test_result.round(3)

### Verification checkpoint

1. Is the test result reasonably consistent with the validation result?
2. What would it mean if the test result were dramatically worse?
3. Why should you not repeatedly change the model after seeing this test result?

## 8. Optimised boosting implementation and early stopping

`HistGradientBoostingClassifier` groups continuous feature values into bins before searching for splits. This can make training more efficient on larger tabular datasets.

It can also stop automatically when its internal validation score no longer improves.

In [ ]:
hist_model = comparison_models['Histogram Gradient Boosting']

print('Iterations actually used:', hist_model.n_iter_)
print('Maximum allowed iterations:', hist_model.max_iter)
print('Stopped early:', hist_model.n_iter_ < hist_model.max_iter)

### Stop and explain

1. Why can binning make split search faster?
2. What is the benefit of early stopping?
3. Why does automatic early stopping still depend on choices such as validation fraction and patience?

## 9. Optional interpretation: permutation importance on validation data

Boosting is often less directly readable than one small decision tree. We can measure how much validation performance decreases when one feature is permuted.

This is evidence of predictive usefulness for this model and dataset. It is not evidence of causality.

In [ ]:
importance = permutation_importance(
    selected_gb,
    X_test,
    y_test,
    scoring='roc_auc',
    n_repeats=15,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_table = pd.DataFrame({
    'feature': X.columns,
    'mean ROC-AUC decrease': importance.importances_mean,
    'standard deviation': importance.importances_std,
}).sort_values('mean ROC-AUC decrease', ascending=False)

display(importance_table.head(10).round(4))

top_importance = importance_table.head(10).sort_values('mean ROC-AUC decrease')
plt.figure(figsize=(9, 5.5))
plt.barh(
    top_importance['feature'],
    top_importance['mean ROC-AUC decrease'],
)
plt.xlabel('Mean ROC-AUC decrease')
plt.ylabel('Feature')
plt.title('Top permutation importances for the selected boosting model')
plt.show()

### Interpretation checkpoint

Write one careful sentence about the top feature.

Avoid:

> “This feature causes the diagnosis.”

Prefer a statement about how much the fitted model relied on the feature for held-out prediction.

## 10. Assessed task — Boosting configuration memo (2 points)

You have a simple constraint:

> **Use at most 300 boosting trees and select the model using validation data.**

Compare at least three configurations. You may change:

- `learning_rate`,
- `n_estimators`,
- `max_depth`,
- `subsample`.

Your memo must include:

1. **Question:** What configuration decision are you making?
2. **Configurations:** At least three candidates.
3. **Evidence:** Validation ROC-AUC and log loss, plus one staged curve.
4. **Diagnosis:** Underfitting, useful learning, or overfitting.
5. **Recommendation:** One configuration and a defensible stage count.
6. **Final audit:** Test-set performance evaluated only after the choice.
7. **Limitation:** One reason the recommendation may not transfer.
8. **AI note:** If AI was used, report one question and one independently verified revision.

### Marking

| Criterion | Points |
|---|---:|
| Configurations and validation comparison are appropriate | 0.75 |
| Recommendation follows from curves and metrics | 0.50 |
| Test set is used only for the final audit | 0.50 |
| Limitation or verified AI note is meaningful | 0.25 |

In [ ]:
# Change configurations here. Keep n_estimators <= 300.
CANDIDATE_CONFIGS = {
    'conservative': {
        'learning_rate': 0.03,
        'n_estimators': 300,
        'max_depth': 1,
        'subsample': 1.0,
    },
    'balanced': {
        'learning_rate': 0.08,
        'n_estimators': 180,
        'max_depth': 1,
        'subsample': 1.0,
    },
    'more complex': {
        'learning_rate': 0.08,
        'n_estimators': 140,
        'max_depth': 2,
        'subsample': 0.80,
    },
}

candidate_rows = []
candidate_models = {}

for name, params in CANDIDATE_CONFIGS.items():
    model = GradientBoostingClassifier(
        random_state=RANDOM_STATE,
        **params,
    )
    model.fit(X_train, y_train)
    candidate_models[name] = model

    probability = model.predict_proba(X_valid)[:, 1]
    prediction = (probability >= 0.5).astype(int)

    candidate_rows.append({
        'configuration': name,
        **params,
        'validation accuracy': accuracy_score(y_valid, prediction),
        'validation F1': f1_score(y_valid, prediction),
        'validation ROC-AUC': roc_auc_score(y_valid, probability),
        'validation log loss': log_loss(y_valid, probability),
    })

candidate_results = pd.DataFrame(candidate_rows).set_index('configuration')
candidate_results.round(3)

In [ ]:
# Choose one candidate only after reviewing validation evidence.
SELECTED_CANDIDATE = 'balanced'

chosen_params = CANDIDATE_CONFIGS[SELECTED_CANDIDATE]
final_candidate = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
    **chosen_params,
)
final_candidate.fit(X_train_full, y_train_full)

final_probability = final_candidate.predict_proba(X_test)[:, 1]
final_prediction = (final_probability >= 0.5).astype(int)

pd.Series({
    'selected configuration': SELECTED_CANDIDATE,
    'test accuracy': accuracy_score(y_test, final_prediction),
    'test F1': f1_score(y_test, final_prediction),
    'test ROC-AUC': roc_auc_score(y_test, final_probability),
    'test log loss': log_loss(y_test, final_probability),
})

### Your boosting configuration memo

**Question:**  

**Configurations compared:**  

**Validation evidence:**  

**Curve diagnosis:**  

**Recommendation:**  

**Final test audit:**  

**Limitation:**  

**Optional verified AI note:**

# End-of-lab self-check

Before finishing, check that you can:

- [ ] explain what residuals represent in boosting,
- [ ] describe one boosting update,
- [ ] explain why shallow trees can form a complex additive model,
- [ ] describe the interaction between learning rate and number of trees,
- [ ] read training and validation stage curves,
- [ ] identify a defensible early-stopping point,
- [ ] compare Random Forest and Gradient Boosting conceptually,
- [ ] explain the purpose of Histogram Gradient Boosting,
- [ ] keep validation and test roles separate,
- [ ] interpret permutation importance without causal language,
- [ ] use AI questions to improve an evidence-based diagnosis.

Exercise 8 will focus on dimensionality reduction and latent-space visualisation.